# TeCo EEG-only: bounded V4–V6 follow-ups
Run sections in order. V3 has its own notebook, `03_v3_preprocessing.ipynb`. V4 has a saved-output audit, two mechanism replays, an eight-extra-fit mini-ablation, and a 180-fit full comparison. V5 has 144 neural fits plus fast controls. V6 is gated until a CNN configuration is selected. No cell silently changes the locked settings.

## Setup
Start a **fresh GPU runtime**. This uses packages already in Colab; it does not install local Mac dependencies.

In [ ]:
import os, sys, subprocess, json
repo = '/content/reproduce-teco-results'
if not os.path.isdir(repo):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/parmisbathayan/reproduce-teco-results.git', repo], check=True)
os.chdir(repo)
from google.colab import drive
drive.mount('/content/drive')
import tensorflow as tf
assert tf.config.list_physical_devices('GPU'), 'Select Runtime > Change runtime type > GPU'
print('commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('TensorFlow:', tf.__version__)


In [ ]:
trt_dir = '/content/drive/MyDrive/Thesis/Data/TRT_Total'
labels_csv = '/content/drive/MyDrive/Thesis/Data/teco_sentiment_labels_task1.csv'
base = '/content/drive/MyDrive/Thesis/Results/reproduce_yaser_codes/eegonly'
config = 'configs/v4_v6_followup.json'
for path in [trt_dir, labels_csv, config]:
    assert os.path.exists(path), f'Missing: {path}'
def run_script(script, out_name, *extra):
    command = [sys.executable, 'src/' + script, '--trt-dir', trt_dir,
               '--labels-csv', labels_csv, '--out-dir', os.path.join(base, out_name),
               '--config', config, *extra]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)


## 1. Saved-output audit and focused mechanism probe
Inspect V2's saved results first. The two fixed probe runs compare seed 45 (collapse-prone) with seed 62 (no V2 collapses) for JAY. V2 model weights were not saved, so these are replays, not direct examination of the old models.

In [ ]:
import pandas as pd
from IPython.display import display
v2_path = os.path.join(base, 'v2_diagnoseCollapse', 'fold_metrics.csv')
v2 = pd.read_csv(v2_path)
assert len(v2) == 60
display(v2.groupby('seed').agg(mean_f1=('f1', 'mean'), collapses=('collapsed', 'sum')))
display(v2[v2.participant == 'JAY'][['seed', 'f1', 'collapsed', 'stopping_epoch', 'best_val_macro_f1']])


In [ ]:
audit_dir = os.path.join(base, 'v4_stability', 'saved_v2_audit')
subprocess.run([sys.executable, 'src/v2_saved_audit.py', '--v2-dir',
                os.path.join(base, 'v2_diagnoseCollapse'), '--out-dir', audit_dir], check=True)
display(pd.read_csv(os.path.join(audit_dir, 'participant_characteristics.csv'))
        [['participant', 'collapse_count', 'mean_macro_f1', 'all_zero_sentences',
          'signal_zero_fraction', 'mean_shift_z_rms', 'zero_fraction_difference']])
display(pd.read_csv(os.path.join(audit_dir, 'probability_audit.csv'))
        .groupby('collapsed').constant_probability_output.agg(['sum', 'count']))


In [ ]:
run_script('v4_stability.py', 'v4_stability', '--stage', 'probe')
display(pd.read_csv(os.path.join(base, 'v4_stability', 'probe', 'probe_summary.csv')))


### Small causal diagnostic (before the full V4 run)
For JAY and ARM at seeds 45 and 62, compare the historical V2 tail+accuracy result with tail+macro-F1, whole-participant+accuracy, and whole-participant+macro-F1. This is deliberately small; it helps distinguish checkpoint from validation effects but is not a population estimate. Four of its fits count toward V4 full and will be skipped there.

In [ ]:
run_script('v4_stability.py', 'v4_stability', '--stage', 'ablation')
display(pd.read_csv(os.path.join(base, 'v4_stability', 'ablation_metrics.csv'))
        [['condition', 'seed', 'participant', 'macro_f1', 'collapsed']])


## 2. V3 preprocessing
Run `notebooks/03_v3_preprocessing.ipynb` in a fresh Colab runtime before drawing conclusions about scaling. It is 240 fits and resumes. Compare its raw control to V2; if the control differs materially, investigate before interpreting the transformed methods.

## 3. V4 subject-wise validation and learning rates
Three predeclared rates × five seeds × 12 outer participants = 180 fits. One of the 11 training participants supplies validation, and checkpointing uses validation macro-F1. This changes validation and checkpointing together versus V2, so do not assign a unique causal explanation to either.

In [ ]:
run_script('v4_stability.py', 'v4_stability', '--stage', 'full')
display(pd.read_csv(os.path.join(base, 'v4_stability', 'summary_by_condition.csv')))


## 4. V5 baselines
Majority, two logistic representations, a small MLP, a smaller CNN, and fixed sentence-label-shuffle controls for both logistic regression and the original CNN (CNN control: seeds 62 and 72 only). A shuffled-label score tests participant-invariant sentence fingerprints, **not** sentiment learning. All use the same outer held-out participants and a common ten-participant fit pool.

In [ ]:
run_script('v5_baselines.py', 'v5_baselines')
display(pd.read_csv(os.path.join(base, 'v5_baselines', 'summary_by_model.csv')))


## 5. V6 paired holdouts — stop for review first
Do not run V6 until we inspect V3–V5 and freeze a CNN configuration in a new JSON copied from `configs/v6_selection_TEMPLATE.json`. Set `status` to `LOCKED` and record the result files and rationale. Its five predeclared pairs × five seeds are 25 fits, with **separate** scores for each held-out participant. Because configuration selection used the same participants in earlier versions, treat V6 as exploratory, not an independent test.

In [ ]:
selection = '/content/drive/MyDrive/Thesis/Results/reproduce_yaser_codes/eegonly/v6_selection_LOCKED.json'
assert os.path.exists(selection), 'Stop: review V3–V5 and create the locked selection first.'
run_script('v6_leave_two_out.py', 'v6_leave_two_out', '--selection', selection)
display(pd.read_csv(os.path.join(base, 'v6_leave_two_out', 'summary_by_pair_participant.csv')))
